# LAB 2: Search Pipeline com RRF
## Busca Hibrida Avancada no OpenSearch

**Objetivo:** Configurar e testar Search Pipelines com RRF e Min-Max para otimizar buscas hibridas.

**Prerequisito:** LAB1 deve ser executado previamente

**Duracao estimada:** 90 minutos

## 1. Configuracao

In [ ]:
import json
import os
from opensearchpy import OpenSearch
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

print("Bibliotecas importadas!")

In [ ]:
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASSWORD = os.getenv('OPENSEARCH_PASSWORD', 'admin')
OPENSEARCH_VERIFY_CERTS = os.getenv('OPENSEARCH_VERIFY_CERTS', 'False').lower() == 'true'

INDEX_NAME = "corpus_juridico_aula4"
MODEL_NAME = "BAAI/bge-m3"

try:
    client = OpenSearch(
        hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
        http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
        use_ssl=True,
        verify_certs=OPENSEARCH_VERIFY_CERTS,
        ssl_show_warn=False,
        http_compress=True
    )
    
    info = client.info()
    print(f"Conectado ao OpenSearch {info['version']['number']}")
except Exception as e:
    print(f"Erro ao conectar: {str(e)}")

In [ ]:
# Verificar LAB1
try:
    count_response = client.count(index=INDEX_NAME)
    doc_count = count_response['count']
    
    if doc_count == 0:
        print(f"Indice vazio! Execute LAB1 primeiro.")
    else:
        print(f"Indice contém {doc_count} documentos - LAB1 OK")
except Exception as e:
    print(f"Indice nao encontrado. Execute LAB1!")

In [ ]:
# Carregar modelo
print(f"Carregando modelo {MODEL_NAME}...")

try:
    model = SentenceTransformer(MODEL_NAME, device='cpu')
    print(f"Modelo carregado!")
except Exception as e:
    print(f"Erro ao carregar modelo: {str(e)}")

## 2. Teoria: RRF e Min-Max

RRF (Reciprocal Rank Fusion): score = 1 / (k + rank) onde k=60

Min-Max Normalization: normalized_score = (score - min) / (max - min)

RRF eh mais robusto para combinar multiplas fontes heterogeneas.

## 3. Criar Search Pipeline com RRF

In [ ]:
pipeline_name_rrf = "hybrid_rrf_pipeline"

rrf_pipeline = {
    "description": "Pipeline para busca hibrida com RRF",
    "processors": [
        {
            "normalization-processor": {
                "normalization": {
                    "technique": "rrf",
                    "rrf": {"k": 60}
                },
                "combination": {
                    "technique": "arithmetic_mean",
                    "parameters": {"weights": [0.3, 0.7]}
                }
            }
        }
    ]
}

print(f"Criando pipeline: {pipeline_name_rrf}")

try:
    response = client.transport.perform_request(
        method="PUT",
        url=f"/_search/pipeline/{pipeline_name_rrf}",
        body=rrf_pipeline
    )
    print(f"Pipeline RRF criado!")
except Exception as e:
    print(f"Erro ao criar pipeline RRF: {str(e)}")

## 4. Criar Search Pipeline com Min-Max

In [ ]:
pipeline_name_minmax = "hybrid_minmax_pipeline"

minmax_pipeline = {
    "description": "Pipeline para busca hibrida com Min-Max",
    "processors": [
        {
            "normalization-processor": {
                "normalization": {"technique": "min_max"},
                "combination": {
                    "technique": "arithmetic_mean",
                    "parameters": {"weights": [0.3, 0.7]}
                }
            }
        }
    ]
}

print(f"Criando pipeline: {pipeline_name_minmax}")

try:
    response = client.transport.perform_request(
        method="PUT",
        url=f"/_search/pipeline/{pipeline_name_minmax}",
        body=minmax_pipeline
    )
    print(f"Pipeline Min-Max criado!")
except Exception as e:
    print(f"Erro ao criar pipeline Min-Max: {str(e)}")

## 5. Executar Buscas Comparativas

In [ ]:
QUERY_TEXT = "prisao preventiva requisitos"

query_embedding = model.encode(QUERY_TEXT, normalize_embeddings=True).tolist()

print(f"Query: {QUERY_TEXT}")
print(f"Embedding gerado com dimensao: {len(query_embedding)}")

In [ ]:
# 1. Busca BM25 Pura
bm25_query = {
    "size": 5,
    "query": {"match": {"text": {"query": QUERY_TEXT}}},
    "_source": ["id", "titulo", "tipo_documento"]
}

try:
    response = client.search(index=INDEX_NAME, body=bm25_query)
    bm25_results = response['hits']['hits']
    
    print("\nBUSCA BM25 (Lexical)")
    print(f"Resultados: {len(bm25_results)}")
    for idx, hit in enumerate(bm25_results, 1):
        print(f"  {idx}. [{hit['_score']:7.4f}] {hit['_source']['titulo']}")
except Exception as e:
    print(f"Erro em busca BM25: {str(e)}")
    bm25_results = []

In [ ]:
# 2. Busca kNN Pura
knn_query = {
    "size": 5,
    "query": {"knn": {"knn_vector": {"vector": query_embedding, "k": 5}}},
    "_source": ["id", "titulo", "tipo_documento"]
}

try:
    response = client.search(index=INDEX_NAME, body=knn_query)
    knn_results = response['hits']['hits']
    
    print("\nBUSCA kNN (Semantic)")
    print(f"Resultados: {len(knn_results)}")
    for idx, hit in enumerate(knn_results, 1):
        print(f"  {idx}. [{hit['_score']:7.4f}] {hit['_source']['titulo']}")
except Exception as e:
    print(f"Erro em busca kNN: {str(e)}")
    knn_results = []

In [ ]:
# 3. Busca Hibrida com RRF
hybrid_rrf_query = {
    "size": 5,
    "query": {
        "hybrid": {
            "queries": [
                {"match": {"text": {"query": QUERY_TEXT}}},
                {"knn": {"knn_vector": {"vector": query_embedding, "k": 5}}}
            ]
        }
    },
    "search_pipeline": pipeline_name_rrf,
    "_source": ["id", "titulo", "tipo_documento"]
}

try:
    response = client.search(index=INDEX_NAME, body=hybrid_rrf_query)
    hybrid_rrf_results = response['hits']['hits']
    
    print("\nBUSCA HIBRIDA com RRF")
    print(f"Resultados: {len(hybrid_rrf_results)}")
    for idx, hit in enumerate(hybrid_rrf_results, 1):
        print(f"  {idx}. [{hit['_score']:7.4f}] {hit['_source']['titulo']}")
except Exception as e:
    print(f"Erro em busca hibrida RRF: {str(e)}")
    hybrid_rrf_results = []

In [ ]:
# 4. Busca Hibrida com Min-Max
hybrid_minmax_query = {
    "size": 5,
    "query": {
        "hybrid": {
            "queries": [
                {"match": {"text": {"query": QUERY_TEXT}}},
                {"knn": {"knn_vector": {"vector": query_embedding, "k": 5}}}
            ]
        }
    },
    "search_pipeline": pipeline_name_minmax,
    "_source": ["id", "titulo", "tipo_documento"]
}

try:
    response = client.search(index=INDEX_NAME, body=hybrid_minmax_query)
    hybrid_minmax_results = response['hits']['hits']
    
    print("\nBUSCA HIBRIDA com Min-Max")
    print(f"Resultados: {len(hybrid_minmax_results)}")
    for idx, hit in enumerate(hybrid_minmax_results, 1):
        print(f"  {idx}. [{hit['_score']:7.4f}] {hit['_source']['titulo']}")
except Exception as e:
    print(f"Erro em busca hibrida Min-Max: {str(e)}")
    hybrid_minmax_results = []

## 6. Tabela Comparativa

In [ ]:
def extract_result_data(results, mode_name):
    data = []
    for rank, hit in enumerate(results[:5], 1):
        titulo = hit['_source']['titulo']
        if len(titulo) > 45:
            titulo = titulo[:45] + '...'
        data.append({
            'Modo': mode_name,
            'Rank': rank,
            'Titulo': titulo,
            'Score': hit['_score']
        })
    return data

all_results = []
all_results.extend(extract_result_data(bm25_results, 'BM25'))
all_results.extend(extract_result_data(knn_results, 'kNN'))
all_results.extend(extract_result_data(hybrid_rrf_results, 'RRF'))
all_results.extend(extract_result_data(hybrid_minmax_results, 'Min-Max'))

df_results = pd.DataFrame(all_results)

print("\nTABELA COMPARATIVA: TOP-5 RESULTADOS")
print(f"Query: {QUERY_TEXT}")
print(df_results.to_string(index=False))

## 7. Analise de Scores

In [ ]:
modes = ['BM25', 'kNN', 'RRF', 'Min-Max']
results_list = [bm25_results, knn_results, hybrid_rrf_results, hybrid_minmax_results]

scores_data = {}
for mode, results in zip(modes, results_list):
    scores_data[mode] = [hit['_score'] for hit in results[:5]]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Scores de Busca - Query: {QUERY_TEXT}', fontsize=14, fontweight='bold')

for idx, (ax, (mode, scores)) in enumerate(zip(axes.flat, scores_data.items())):
    if len(scores) > 0:
        ranks = list(range(1, len(scores) + 1))
        colors = plt.cm.viridis(np.linspace(0, 1, len(scores)))
        
        bars = ax.bar(ranks, scores, color=colors, alpha=0.7, edgecolor='black')
        
        ax.set_title(f'{mode}', fontweight='bold')
        ax.set_xlabel('Rank')
        ax.set_ylabel('Score')
        ax.set_xticks(ranks)
        ax.grid(axis='y', alpha=0.3)
        
        for bar, score in zip(bars, scores):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{score:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("Grafico de scores gerado")

## 8. Analise: Lexical vs Semantic

In [ ]:
print("\nANALISE COMPARATIVA\n")
print("Query Lexical (palavras-chave): 'habeas corpus'")
print("Query Semantic (conceito): 'protecao contra privacao de liberdade'")
print("\nObservacao:")
print("- Query lexical = melhor com BM25")
print("- Query semantic = melhor com kNN")
print("- Query desconhecida = usar Busca Hibrida (RRF)")

In [ ]:
# Teste com query lexical
query_text_lex = "habeas corpus"
query_embedding_lex = model.encode(query_text_lex, normalize_embeddings=True).tolist()

bm25_lex = client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {"match": {"text": {"query": query_text_lex}}},
        "_source": ["titulo"]
    }
)['hits']['hits']

knn_lex = client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {"knn": {"knn_vector": {"vector": query_embedding_lex, "k": 3}}},
        "_source": ["titulo"]
    }
)['hits']['hits']

print(f"\nQuery Lexical: '{query_text_lex}'")
print(f"\nBM25 (melhor esperado):")
for rank, hit in enumerate(bm25_lex, 1):
    print(f"  {rank}. [{hit['_score']:.4f}] {hit['_source']['titulo']}")

print(f"\nkNN:")
for rank, hit in enumerate(knn_lex, 1):
    print(f"  {rank}. [{hit['_score']:.4f}] {hit['_source']['titulo']}")

## 9. Exercicio Proposto

Tarefa: Criar 3 novos pipelines com pesos diferentes:
- Pipeline 1: [0.5, 0.5] - Peso igual
- Pipeline 2: [0.7, 0.3] - Priorizar BM25
- Pipeline 3: [0.2, 0.8] - Priorizar kNN

Testar com multiplas queries e comparar resultados.

In [ ]:
print("Implemente o exercicio aqui")

## Referencias ABNT

BAAI. OpenAI/Sentence-Transformers. Disponivel em: https://huggingface.co/BAAI/bge-m3. Acesso em: 16 abr. 2024.

CORMACK, G. V.; CLARKE, C. L. A.; BUETTCHER, S. Reciprocal Rank Fusion Outperforms Condorcet. SIGIR FORUM, v. 43, n. 1, p. 1-8, 2009.

OPENSEARCH. OpenSearch Documentation: Search Pipelines. Disponivel em: https://opensearch.org/docs/latest/search-plugins/search-pipelines/. Acesso em: 16 abr. 2024.

SENTENCE-TRANSFORMERS. Semantic Search. Disponivel em: https://www.sbert.net/docs/usage/semantic_search.html. Acesso em: 16 abr. 2024.